In [1]:
import polars as pl
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_absolute_error, root_mean_squared_error

from catboost import CatBoostRegressor

import warnings

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_parquet('/Users/egor/VS_GIT_repositories/BYTE/src/data/tabular/ya_realty_with_txt_embeds.parquet')
df.shape

(30006, 23)

In [3]:
df.head()

,offer_id,price,price_numeric,old_price,area,rooms,floor,price_per_m2,metro,metro_time,...,photo_count,badges,publish_date,url,title,description,image_urls,self_floor,max_floor,description_embedding
0,7035113340557126091,7 500 000 ₽,7500000.0,NaN,17.7,студия,9 этаж из 16,None,Калитники,9.0,...,1.0,None,None,https://realty.yandex.ru/offer/703511334055712...,апартаменты-студия,Номер лота: 99696. Панорамный вид из больших о...,https://avatars.mds.yandex.net/get-realty-offe...,9.0,16.0,"[0.06329139, 0.07912303, -0.01971404, 0.040010..."
1,7035113340416809607,7 500 000 ₽,7500000.0,NaN,17.0,студия,2 этаж из 2,None,Соколиная гора,8.0,...,1.0,None,6 часов назад назад,https://realty.yandex.ru/offer/703511334041680...,апартаменты-студия,Номер лота: 87440. Продается студия с дизайнер...,https://avatars.mds.yandex.net/get-realty-offe...,2.0,2.0,"[0.056245465, 0.047387548, -0.03474806, 0.0382..."
2,7053956964805047621,12 200 000 ₽,12200000.0,NaN,17.9,студия,2 этаж из 48,None,Тушинская,10.0,...,1.0,None,3 квартал 2027,https://realty.yandex.ru/offer/705395696480504...,квартира-студия,"Арт. 119802099 Студия 17,9 м в CITYZEN Урбан-б...",https://avatars.mds.yandex.net/get-realty-offe...,2.0,48.0,"[0.04167684, 0.04920646, -0.018764082, 0.01812..."
3,7053956914445503237,7 300 000 ₽,7300000.0,NaN,15.7,студия,5 этаж из 5,None,Бутырская,17.0,...,1.0,None,6 часов назад назад,https://realty.yandex.ru/offer/705395691444550...,квартира-студия,Арт. 134010491 СПЕЦИАЛЬНО для наших клиентов с...,https://avatars.mds.yandex.net/get-realty-offe...,5.0,5.0,"[0.070305124, 0.027462784, -0.005403769, -0.00..."
4,3699730400767130013,10 802 031 ₽,10802031.0,NaN,14.1,студия,5 этаж из 16,None,Коммунарка,14.0,...,1.0,None,2 квартал 2026,https://realty.yandex.ru/offer/369973040076713...,квартира-студия,Строим кварталы для жизни с заботой о будущем....,https://avatars.mds.yandex.net/get-realty-offe...,5.0,16.0,"[0.07480104, 0.024318233, -0.016136346, 0.0001..."


In [10]:
X = df[['area', 'metro_time', 'photo_count', 
'self_floor', 'max_floor', 'description_embedding']]

Y = df['price_numeric']

X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, shuffle=True)

In [11]:
model = CatBoostRegressor(
    iterations=4000,
    learning_rate=0.05,
    depth=5,
    l2_leaf_reg=1,
    # cat_features=['metro', 'title', 'author'],
    # text_features=['description', 'address', 'metro'],
    embedding_features=['description_embedding'],
    verbose=200
)

model.fit(X_train, Y_train)

0:	learn: 10514549.0630897	total: 65.6ms	remaining: 4m 22s
200:	learn: 6564167.0446606	total: 468ms	remaining: 8.84s
400:	learn: 6070656.9067759	total: 866ms	remaining: 7.77s
600:	learn: 5653458.1975230	total: 1.35s	remaining: 7.64s
800:	learn: 5381796.9550040	total: 2.16s	remaining: 8.63s
1000:	learn: 5157075.2426066	total: 2.89s	remaining: 8.65s
1200:	learn: 4992300.0936959	total: 3.5s	remaining: 8.15s
1400:	learn: 4848632.6289078	total: 4.2s	remaining: 7.79s
1600:	learn: 4692902.9240432	total: 5.51s	remaining: 8.26s
1800:	learn: 4559673.9720347	total: 6.51s	remaining: 7.95s
2000:	learn: 4456413.1627154	total: 7.93s	remaining: 7.93s
2200:	learn: 4361180.9595163	total: 9.57s	remaining: 7.82s
2400:	learn: 4287094.7524398	total: 11s	remaining: 7.35s
2600:	learn: 4216678.3861824	total: 12.1s	remaining: 6.49s
2800:	learn: 4147279.2064876	total: 13.3s	remaining: 5.71s
3000:	learn: 4077865.3052485	total: 14s	remaining: 4.66s
3200:	learn: 4003177.1313729	total: 15.1s	remaining: 3.77s
3400:	l

CatBoostRegressor(depth=5, embedding_features=['description_embedding'], iterations=4000, l2_leaf_reg=1, learning_rate=0.05, loss_function='RMSE', verbose=200)

In [12]:
def eval_with_metrics(model, X, Y):
    
    preds = model.predict(X)

    print(f"R^2: {r2_score(Y, preds)} \n"
          f"MAE: {mean_absolute_error(Y, preds)} \n"
          f"MAPE: {mean_absolute_percentage_error(Y, preds)} \n"
          f"RMSE: {root_mean_squared_error(Y, preds)} \n")
    
eval_with_metrics(model, X_val, Y_val)

R^2: 0.7494379845705381 
MAE: 2771576.366002571 
MAPE: 0.22889700813757488 
RMSE: 5384279.010243824 



In [ ]:
X = df[['area', 'metro_time', 'photo_count', 
'self_floor', 'max_floor', 'metro', 'title', 
'author', 'description', 'address', 'description_embedding']]

X['metro'] = X['metro'].apply(lambda x: str(x))
X['title'] = X['title'].apply(lambda x: str(x))
X['author'] = X['author'].apply(lambda x: str(x))
X['description'] = X['description'].apply(lambda x: str(x))
X['address'] = X['address'].apply(lambda x: str(x))


Y = df['price_numeric']

X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, shuffle=True)

In [ ]:
model_full = CatBoostRegressor(
    iterations=4000,
    learning_rate=0.05,
    depth=5,
    l2_leaf_reg=1,
    cat_features=['metro', 'title', 'author'],
    text_features=['description', 'address', 'metro'],
    embedding_features=['description_embedding'],
    verbose=200
)

model_full.fit(X_train, Y_train)

In [ ]:
def eval_with_metrics(model, X, Y):
    
    preds = model.predict(X)

    print(f"R^2: {r2_score(Y, preds)} \n"
          f"MAE: {mean_absolute_error(Y, preds)} \n"
          f"MAPE: {mean_absolute_percentage_error(Y, preds)} \n"
          f"RMSE: {root_mean_squared_error(Y, preds)} \n")
    
eval_with_metrics(model_full, X_val, Y_val)